# 08 — Runtime Engine Verification & Paired Metrics

Notebook này đọc **engineering smoke outputs** đã sinh bởi DES engine thực tế.

> Các output hiện tại dùng `haversine_smoke_test`, nên chỉ chứng minh pipeline chạy đúng về mặt phần mềm/invariant. Không dùng các số này làm kết quả khoa học cuối cùng.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,cwd.parent,*cwd.parents] if (p/'experiments'/'smoke_3rep').exists()),None)
if ROOT is None: raise FileNotFoundError('Không tìm thấy experiments/smoke_3rep')
EXP=ROOT/'experiments'/'smoke_3rep'
metrics=pd.read_csv(EXP/'replication_metrics.csv')
validation=json.loads((EXP/'runtime_validation.json').read_text())
print('ROOT=',ROOT)
print('rows=',len(metrics))
validation

## 1. Pair completeness

In [ ]:
pair=(metrics.groupby(['scenario_id','replication_id'])
      .agg(policy_count=('policy_id','nunique'),seed_count=('random_seed','nunique'),group_count=('common_random_group','nunique'))
      .reset_index())
pair['complete']=(pair.policy_count==4)&(pair.seed_count==1)&(pair.group_count==1)
pair

## 2. Mean metric by policy — smoke only

In [ ]:
show=['served_rate','feasible_rate_overall','SLA_pass_rate','avg_response_time_min','P95_response_P1']
display(metrics.groupby(['scenario_id','policy_id'])[show].mean().round(3))

In [ ]:
for metric in ['feasible_rate_overall','SLA_pass_rate','avg_response_time_min']:
    pivot=metrics.pivot_table(index='scenario_id',columns='policy_id',values=metric,aggfunc='mean')
    ax=pivot.plot(kind='bar',figsize=(9,4.5),title=f'{metric} — 3-rep smoke test')
    ax.set_ylabel(metric)
    ax.grid(axis='y',alpha=.25)
    plt.xticks(rotation=0)
    plt.show()

## 3. B6 solver diagnostics

In [ ]:
b6=metrics[metrics.policy_id=='B6']
display(b6[['scenario_id','replication_id','avg_solver_time_sec','max_mip_gap','b6_epsilon_relaxation_count','routing_source']])
print('Mean B6 solver time/epoch:',b6.avg_solver_time_sec.mean())
print('Max MIP gap:',b6.max_mip_gap.max())
print('Total epsilon relaxations:',b6.b6_epsilon_relaxation_count.sum())

## 4. Runtime invariant audit

In [ ]:
pd.DataFrame(validation['checks'])[['check','passed','detail']]

## 5. Final 30-paired command

Chỉ chạy sau khi Notebook 02 đã tạo `data/osmnx/*`:

```bash
./scripts/run_30_paired_osmnx.sh
```

Không thêm `--allow-haversine-fallback` vào experiment cuối.